In [1]:
import numpy as np
import pandas as pd
from math import pi
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
import statsmodels.formula.api as smf
import sklearn.linear_model as sklm
import matplotlib.pyplot as plt

In [2]:
data = pd.read_excel('./data/1EData_PredictorData2019.xlsx', sheet_name='Monthly')
data

,yyyymm,Index,D12,E12,b/m,tbl,AAA,BAA,lty,ntis,Rfree,infl,ltr,corpr,svar,csp,CRSP_SPvw,CRSP_SPvwx,PPIG,IPG
0,187101,4.440000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,0.004967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,187102,4.500000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,0.004525,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,187103,4.610000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,0.004252,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,187104,4.740000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,0.004643,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,187105,4.860000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,0.003698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1783,201908,2926.459961,56.838763,133.69,0.237917,0.0210,0.0298,0.0387,0.0163,-0.010244,0.001625,-0.000051,0.0797,0.0738,0.004318,NaN,-0.016085,-0.018377,-0.747384,0.705045
1784,201909,2976.739990,57.219507,132.90,0.233377,0.0195,0.0303,0.0391,0.0170,-0.010959,0.001575,0.000783,-0.0192,-0.0190,0.000605,NaN,0.018791,0.017272,-0.401606,-0.347551
1785,201910,3037.560059,57.559879,135.09,0.232261,0.0189,0.0301,0.0392,0.0171,-0.013267,0.001375,0.002286,-0.0052,0.0006,0.001510,NaN,0.021621,0.020441,0.100806,-0.406952
1786,201911,3140.979980,57.900251,137.28,0.223938,0.0165,0.0306,0.0394,0.0181,-0.007907,0.001283,-0.000536,-0.0059,0.0014,0.000306,NaN,0.036206,0.033979,0.201410,0.928027


In [3]:
data['DP'] = data['D12'].apply(np.log) - data['Index'].apply(np.log)  #DP（股息价格比）
data['EP'] = data['E12'].apply(np.log) - data['Index'].apply(np.log)  #EP（盈利价格比）
data['VOL'] = data['CRSP_SPvw'].abs().rolling(window=12).mean()*np.sqrt(pi/6) #VOL（波动率）
# 用于指定一个滚动窗口，在这个例子中窗口大小为12。这意味着对于时间序列中的每个点，它将考虑该点之前和包括该点在内的共12个数据点。
data['BILL'] = data['tbl'] - data['tbl'].rolling(window=12).mean() #BILL（3个月国库券收益率）
data['BOND'] = data['lty'] - data['lty'].rolling(window=12).mean() #BOND（10年期国债收益率）
data['TERM'] = data['lty'] - data['tbl']  #TERM（期限利差）
data['CREDIT'] = data['AAA'] - data['lty'] # CREDIT（信用利差）
data['MA112'] = data['Index'] >= data['Index'].rolling(window=12).mean()
data['MA312']  =data['Index'].rolling(window=3).mean() >= data['Index'].rolling(window=12).mean()
data['MOM6'] = data['Index'] >= data['Index'].shift(periods=6)
data['ExRet'] = data['CRSP_SPvw'] - data['Rfree'] #ExRet（超额收益）
data[['MA112', 'MA312', 'MOM6']] = data[['MA112', 'MA312', 'MOM6']].astype(int)
data

,yyyymm,Index,D12,E12,b/m,tbl,AAA,BAA,lty,ntis,...,EP,VOL,BILL,BOND,TERM,CREDIT,MA112,MA312,MOM6,ExRet
0,187101,4.440000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,...,-2.406945,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN
1,187102,4.500000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,...,-2.420368,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN
2,187103,4.610000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,...,-2.444519,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN
3,187104,4.740000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,...,-2.472328,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN
4,187105,4.860000,0.260000,0.40,NaN,NaN,NaN,NaN,NaN,NaN,...,-2.497329,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1783,201908,2926.459961,56.838763,133.69,0.237917,0.0210,0.0298,0.0387,0.0163,-0.010244,...,-3.086025,0.031396,-0.001725,-0.009958,-0.0047,0.0135,1,1,1,-0.017710
1784,201909,2976.739990,57.219507,132.90,0.233377,0.0195,0.0303,0.0391,0.0170,-0.010959,...,-3.108987,0.032219,-0.003158,-0.007892,-0.0025,0.0133,1,1,1,0.017216
1785,201910,3037.560059,57.559879,135.09,0.232261,0.0189,0.0301,0.0392,0.0171,-0.013267,...,-3.112869,0.029398,-0.003558,-0.006283,-0.0018,0.0130,1,1,1,0.020246
1786,201911,3140.979980,57.900251,137.28,0.223938,0.0165,0.0306,0.0394,0.0181,-0.007907,...,-3.130267,0.030390,-0.005458,-0.004150,0.0016,0.0125,1,1,1,0.034923


In [4]:
data = pd.concat([data[['yyyymm', 'CRSP_SPvw', 'Rfree', 'ExRet',
                        'DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG', 'IPG',
                        'MA112', 'MA312', 'MOM6']],
                  data[['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG', 'IPG',
                        'MA112', 'MA312', 'MOM6']].shift(periods=1)], axis=1)
data
# L1 lag1 表示滞后一期
#将除了日期列（'yyyymm'）以外的所有列下移一个时间周期。这样做的目的是为了在进行时间序列预测时，预测变量可以相对于目标变量（即未来的收益率）有一个时间上的滞后。这是时间序列分析中的常见做法，因为你不能用当前或未来的信息来预测过去或当前的值。

,yyyymm,CRSP_SPvw,Rfree,ExRet,DP,EP,VOL,BILL,BOND,TERM,...,VOL,BILL,BOND,TERM,CREDIT,PPIG,IPG,MA112,MA312,MOM6
0,187101,NaN,0.004967,NaN,-2.837728,-2.406945,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,187102,NaN,0.004525,NaN,-2.851151,-2.420368,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
2,187103,NaN,0.004252,NaN,-2.875302,-2.444519,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
3,187104,NaN,0.004643,NaN,-2.903111,-2.472328,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
4,187105,NaN,0.003698,NaN,-2.928112,-2.497329,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1783,201908,-0.016085,0.001625,-0.017710,-3.941330,-3.086025,0.031396,-0.001725,-0.009958,-0.0047,...,0.032412,-0.000908,-0.006742,-0.0011,0.0123,0.199700,-0.175883,1.0,1.0,1.0
1784,201909,0.018791,0.001575,0.017216,-3.951689,-3.108987,0.032219,-0.003158,-0.007892,-0.0025,...,0.031396,-0.001725,-0.009958,-0.0047,0.0135,-0.747384,0.705045,1.0,1.0,1.0
1785,201910,0.021621,0.001375,0.020246,-3.965984,-3.112869,0.029398,-0.003558,-0.006283,-0.0018,...,0.032219,-0.003158,-0.007892,-0.0025,0.0133,-0.401606,-0.347551,1.0,1.0,1.0
1786,201911,0.036206,0.001283,0.034923,-3.993568,-3.130267,0.030390,-0.005458,-0.004150,0.0016,...,0.029398,-0.003558,-0.006283,-0.0018,0.0130,0.100806,-0.406952,1.0,1.0,1.0


In [5]:
data.columns = ['yyyymm', 'Ret', 'Rfree', 'ExRet',
                'DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG', 'IPG',
                'MA112', 'MA312', 'MOM6', 'DPL1',
                'EPL1', 'VOLL1', 'BILLL1', 'BONDL1', 'TERML1', 'CREDITL1', 'PPIGL1', 'IPGL1',
                'MA112L1', 'MA312L1', 'MOM6L1']
data
#新的列名中，带有'L1'后缀的列表示这些列是滞后一期的数据。这样做可以确保在后续的分析中，我们清楚地知道哪些变量是原始值，哪些是滞后值。

,yyyymm,Ret,Rfree,ExRet,DP,EP,VOL,BILL,BOND,TERM,...,VOLL1,BILLL1,BONDL1,TERML1,CREDITL1,PPIGL1,IPGL1,MA112L1,MA312L1,MOM6L1
0,187101,NaN,0.004967,NaN,-2.837728,-2.406945,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,187102,NaN,0.004525,NaN,-2.851151,-2.420368,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
2,187103,NaN,0.004252,NaN,-2.875302,-2.444519,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
3,187104,NaN,0.004643,NaN,-2.903111,-2.472328,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
4,187105,NaN,0.003698,NaN,-2.928112,-2.497329,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1783,201908,-0.016085,0.001625,-0.017710,-3.941330,-3.086025,0.031396,-0.001725,-0.009958,-0.0047,...,0.032412,-0.000908,-0.006742,-0.0011,0.0123,0.199700,-0.175883,1.0,1.0,1.0
1784,201909,0.018791,0.001575,0.017216,-3.951689,-3.108987,0.032219,-0.003158,-0.007892,-0.0025,...,0.031396,-0.001725,-0.009958,-0.0047,0.0135,-0.747384,0.705045,1.0,1.0,1.0
1785,201910,0.021621,0.001375,0.020246,-3.965984,-3.112869,0.029398,-0.003558,-0.006283,-0.0018,...,0.032219,-0.003158,-0.007892,-0.0025,0.0133,-0.401606,-0.347551,1.0,1.0,1.0
1786,201911,0.036206,0.001283,0.034923,-3.993568,-3.130267,0.030390,-0.005458,-0.004150,0.0016,...,0.029398,-0.003558,-0.006283,-0.0018,0.0130,0.100806,-0.406952,1.0,1.0,1.0


In [6]:
data = data[data['yyyymm'] >= 192701]
data.reset_index(drop=True, inplace=True)
data

,yyyymm,Ret,Rfree,ExRet,DP,EP,VOL,BILL,BOND,TERM,...,VOLL1,BILLL1,BONDL1,TERML1,CREDITL1,PPIGL1,IPGL1,MA112L1,MA312L1,MOM6L1
0,192701,-0.002910,0.002692,-0.005602,-2.942374,-2.374773,0.022268,-0.001625,-0.001508,0.0044,...,0.022200,0.000808,-0.001400,0.0019,0.0114,-0.588235,-0.400104,1.0,1.0,1.0
1,192702,0.045522,0.002742,0.042780,-2.979535,-2.430353,0.023005,0.000192,-0.001700,0.0024,...,0.022268,-0.001625,-0.001508,0.0044,0.0115,-2.958580,-0.401711,1.0,1.0,1.0
2,192703,0.007324,0.002667,0.004657,-2.976535,-2.445079,0.019967,0.000700,-0.002967,0.0002,...,0.023005,0.000192,-0.001700,0.0024,0.0120,1.219512,0.806663,1.0,1.0,1.0
3,192704,0.013021,0.002825,0.010196,-2.984225,-2.471309,0.018429,-0.000250,-0.002475,0.0013,...,0.019967,0.000700,-0.002967,0.0002,0.0131,-0.602410,1.200312,1.0,1.0,1.0
4,192705,0.062353,0.002775,0.059578,-3.025963,-2.531446,0.021368,0.001392,-0.002725,-0.0012,...,0.018429,-0.000250,-0.002475,0.0013,0.0125,-1.212121,-2.372151,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1111,201908,-0.016085,0.001625,-0.017710,-3.941330,-3.086025,0.031396,-0.001725,-0.009958,-0.0047,...,0.032412,-0.000908,-0.006742,-0.0011,0.0123,0.199700,-0.175883,1.0,1.0,1.0
1112,201909,0.018791,0.001575,0.017216,-3.951689,-3.108987,0.032219,-0.003158,-0.007892,-0.0025,...,0.031396,-0.001725,-0.009958,-0.0047,0.0135,-0.747384,0.705045,1.0,1.0,1.0
1113,201910,0.021621,0.001375,0.020246,-3.965984,-3.112869,0.029398,-0.003558,-0.006283,-0.0018,...,0.032219,-0.003158,-0.007892,-0.0025,0.0133,-0.401606,-0.347551,1.0,1.0,1.0
1114,201911,0.036206,0.001283,0.034923,-3.993568,-3.130267,0.030390,-0.005458,-0.004150,0.0016,...,0.029398,-0.003558,-0.006283,-0.0018,0.0130,0.100806,-0.406952,1.0,1.0,1.0


In [7]:
# 样本内检验
# 单因子模型：OLS线性拟合
factor = 'DP' #设置当前分析的预测因子名称
model = smf.ols('ExRet ~ DPL1', data=data[['ExRet', 'DPL1']])
# 'ExRet ~ DPL1'指定了因变量（ExRet，即超额收益）和自变量（DPL1，即滞后一期的股息价格比）
#  data[['ExRet', 'DPL1']]是用于拟合模型的数据集
results = model.fit()
rg_con = results.params['Intercept']
rg_con_pvalue = results.pvalues['Intercept']
# rg_con 和 rg_con_pvalue：分别存储截距项的估计值和p值。
rg_DP = results.params['DPL1']
rg_DP_pvalue = results.pvalues['DPL1']
# rg_DP 和 rg_DP_pvalue：分别存储自变量DPL1的估计值和p值
if rg_DP_pvalue <= 0.01:
    jud = '在1%的显著性水平下有样本内预测能力'
elif (rg_DP_pvalue > 0.01) & (rg_DP_pvalue <= 0.05):
    jud = '在5%的显著性水平下有样本内预测能力'
elif (rg_DP_pvalue > 0.05) & (rg_DP_pvalue <= 0.1):
    jud = '在10%的显著性水平下有样本内预测能力'
else:
    jud = '无样本内预测能力'
print('In-sample tests for one factor model with OLS:')
print('Predictor: {:s}'.format(factor))
print('Regressing Results: b = {:f}, k = {:f}'.format(rg_con, rg_DP))
print('Regressing Pvalues: p = {:f}, p = {:f}'.format(rg_con_pvalue, rg_DP_pvalue))
print('Inference: {:s}'.format(jud))

#这段代码是用于进行单因子模型的样本内检验，即检验某个特定的预测因子（在这个例子中是DP，即股息价格比）
#是否对超额收益（ExRet）有显著的预测能力。这是通过使用普通最小二乘法（OLS）线性拟合来实现的。

In-sample tests for one factor model with OLS:
Predictor: DP
Regressing Results: b = 0.029255, k = 0.006683
Regressing Pvalues: p = 0.014418, p = 0.056033
Inference: 在10%的显著性水平下有样本内预测能力


In [8]:
# 12个因子的样本内检验函数封装
# 需要检验的因子列表
factors = ['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG', 'IPG', 'MA112', 'MA312', 'MOM6']

for factor in factors:
    factorL1 = factor + 'L1'
    formula = 'ExRet ~ ' + factorL1
    model = smf.ols(formula=formula, data=data[['ExRet',factorL1]])  # 构建模型
    results = model.fit()
    rg_con = results.params['Intercept']
    rg_con_pvalue = results.pvalues['Intercept']
    rg_factor = results.params[factorL1]
    rg_factor_pvalue = results.pvalues[factorL1]
    
    if rg_factor_pvalue <= 0.01:
        jud = '在1%的显著性水平下有样本内预测能力'
    elif (rg_factor_pvalue > 0.01) & (rg_factor_pvalue <= 0.05):
        jud = '在5%的显著性水平下有样本内预测能力'
    elif (rg_factor_pvalue > 0.05) & (rg_factor_pvalue <= 0.1):
        jud = '在10%的显著性水平下有样本内预测能力'
    else:
        jud = '无样本内预测能力'
    print('In-sample tests for one factor model with OLS:')
    print('Predictor: {:s}'.format(factor))
    print('Regressing Results: b = {:f}, k = {:f}'.format(rg_con, rg_factor))
    print('Regressing Pvalues: p = {:f}, p = {:f}'.format(rg_con_pvalue, rg_factor_pvalue))
    print('Inference: {:s}'.format(jud))
    print("-----------------------------------------------------------------------")

In-sample tests for one factor model with OLS:
Predictor: DP
Regressing Results: b = 0.029255, k = 0.006683
Regressing Pvalues: p = 0.014418, p = 0.056033
Inference: 在10%的显著性水平下有样本内预测能力
-----------------------------------------------------------------------
In-sample tests for one factor model with OLS:
Predictor: EP
Regressing Results: b = 0.028835, k = 0.008089
Regressing Pvalues: p = 0.007916, p = 0.038502
Inference: 在5%的显著性水平下有样本内预测能力
-----------------------------------------------------------------------
In-sample tests for one factor model with OLS:
Predictor: VOL
Regressing Results: b = 0.002326, k = 0.154138
Regressing Pvalues: p = 0.493565, p = 0.149339
Inference: 无样本内预测能力
-----------------------------------------------------------------------
In-sample tests for one factor model with OLS:
Predictor: BILL
Regressing Results: b = 0.006617, k = -0.241740
Regressing Pvalues: p = 0.000049, p = 0.254856
Inference: 无样本内预测能力
-----------------------------------------------------------

In [9]:
# myfun_stat_gains函数通过计算和比较预测模型和基准模型的预测误差，评估了预测模型在样本外的统计显著性
def myfun_stat_gains(rout, rmean, rreal):
    #rout是模型的预测收益率，rmean是平均收益率（通常作为基准预测），rreal是实际的收益率
    R2os = 1 - np.sum((rreal-rout)**2)/np.sum((rreal-rmean)**2)
    #如果R2os大于0，这意味着预测模型比简单的平均模型能更好地解释收益率的变动
    d = (rreal - rmean)**2 - ((rreal-rout)**2 - (rmean-rout)**2)
    x = sm.add_constant(np.arange(len(d))+1)
    model = sm.OLS(d, x)
    fitres = model.fit()
    MFSEadj = fitres.tvalues[0]   #mfse的值
    pvalue_MFSEadj = fitres.pvalues[0]   #msfe的p值
    # MFSEadj是回归系数的t统计量，用于检验预测模型是否显著优于基准模型。pvalue_MFSEadj是对应的p值，用于判断统计显著性。

    if (R2os > 0) & (pvalue_MFSEadj <= 0.01):
        jud = '在1%的显著性水平下有样本外预测能力'
    elif (R2os > 0) & (pvalue_MFSEadj > 0.01) & (pvalue_MFSEadj <= 0.05):
        jud = '在5%的显著性水平下有样本外预测能力'
    elif (R2os > 0) & (pvalue_MFSEadj > 0.05) & (pvalue_MFSEadj <= 0.1):
        jud = '在10%的显著性水平下有样本外预测能力'
    else:
        jud = '无样本外预测能力'
    print('Stat gains: R2os = {:f}, MFSEadj = {:f}, MFSEpvalue = {:f}'.format(R2os, MFSEadj, pvalue_MFSEadj))
    print('Inference: {:s}'.format(jud))

    return R2os, MFSEadj, pvalue_MFSEadj

In [10]:
# 目的是评估一个预测模型在经济上的意义
# rout是模型的预测收益率，rmean是平均收益率（作为基准预测），rreal是实际的收益率，rfree是无风险利率，volt2是预测期间的方差，gmm是投资者的风险厌恶系数，其默认值为5。
# 均值模型，上面那个是预测模型
def myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm = 5):
    omg_out = rout/volt2/gmm #在外样本期间的超额收益率
    rp_out = rfree + omg_out*rreal # 在外样本期间的组合收益率
    Uout = np.mean(rp_out) - 0.5*gmm*np.var(rp_out) #Uout是根据投资者的风险厌恶系数gmm计算的预期效用
    omg_mean = rmean/volt2/gmm #在平均情况下的超额收益率
    rp_mean = rfree + omg_mean*rreal #在平均情况下的组合收益率
    Umean = np.mean(rp_mean) - 0.5*gmm*np.var(rp_mean) #在平均情况下的预期效用
    DeltaU = Uout - Umean #计算DeltaU，即预测模型与平均模型之间的效用差异
    #如果DeltaU接近于0（小于一个很小的阈值），则认为预测模型没有经济意义。否则，认为模型具有经济意义。
    if DeltaU < 10**-6:
        jud = '没有经济意义'
    else:
        jud = '有经济意义'
    print('Econ Gains: Delta U = {:f}, Upred = {:f}, Umean = {:f}'.format(DeltaU, Uout, Umean))
    print('Inference: {:s}'.format(jud))

    return Uout, Umean, DeltaU

In [11]:
# 样本外检验
# 单因子模型: OLS线性拟合
factor_out = 'DP'
datafit = data[['yyyymm', 'Ret', 'Rfree', 'ExRet', 'DP', 'DPL1']].copy(deep=True)
#从原始数据中选择相关列并复制，以便进行样本外检验
n_in = np.sum(datafit['yyyymm'] <= 195612)
n_out = np.sum(datafit['yyyymm'] > 195612)
#分别计算样本内和样本外的数据点数量
rout = np.zeros(n_out)
rmean = np.zeros(n_out)
rreal = np.zeros(n_out)
rfree = np.zeros(n_out)
volt2 = np.zeros(n_out)


for i in range(n_out):
    model = smf.ols('ExRet ~ DPL1', data=datafit[['ExRet', 'DPL1']].iloc[:(n_in+i),:])
    results = model.fit()
    b = results.params['Intercept']
    k = results.params['DPL1']
    f = datafit['DP'].iloc[n_in+i-1]
    rreal[i] = datafit['ExRet'].iloc[n_in+i]
    rfree[i] = datafit['Rfree'].iloc[n_in+i]
    rout[i] = k*f+b
    rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
    volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)

print()
print('Out-of-sample tests for one factor model with OLS:')
print('Predictor: {:s}'.format(factor_out))
R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
del datafit


Out-of-sample tests for one factor model with OLS:
Predictor: DP
Stat gains: R2os = -0.005143, MFSEadj = 2.062252, MFSEpvalue = 0.039526
Inference: 无样本外预测能力
Econ Gains: Delta U = -0.000301, Upred = 0.003771, Umean = 0.004072
Inference: 没有经济意义


In [12]:
# 单因子样本外检验的封装
factors = ['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG', 'IPG', 'MA112', 'MA312', 'MOM6']

def perform_out_of_sampleTest(factor,data):
    factorL1 = factor + 'L1'
    datafit = data[['yyyymm','Ret','Rfree','ExRet',factor,factorL1]].copy(deep=True)
    n_in = np.sum(datafit['yyyymm']<=195612)
    n_out = np.sum(datafit['yyyymm']>195612)
    rout = np.zeros(n_out)
    rmean = np.zeros(n_out)
    rreal = np.zeros(n_out)
    rfree = np.zeros(n_out)
    volt2 = np.zeros(n_out)
    
    for i in range(n_out):
        model = smf.ols('ExRet~'+factorL1,data=datafit[['ExRet',factorL1]].iloc[:(n_in+i),:])
        results = model.fit()
        b = results.params['Intercept']
        k = results.params[factorL1]
        f = datafit[factor].iloc[n_in+i-1]
        rreal[i] = datafit['ExRet'].iloc[n_in+i]
        rfree[i] = datafit['Rfree'].iloc[n_in+i]
        rout[i] = k*f+b
        rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
        volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)
        
    print()
    print('Out-of-sample tests for one factor model with OLS:')
    print('Predictor: {:s}'.format(factor))
    R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
    Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
    print('---------------------------------------------------------------------')
    del datafit

for factor in factors:
    perform_out_of_sampleTest(factor,data)


Out-of-sample tests for one factor model with OLS:
Predictor: DP
Stat gains: R2os = -0.005143, MFSEadj = 2.062252, MFSEpvalue = 0.039526
Inference: 无样本外预测能力
Econ Gains: Delta U = -0.000301, Upred = 0.003771, Umean = 0.004072
Inference: 没有经济意义
---------------------------------------------------------------------

Out-of-sample tests for one factor model with OLS:
Predictor: EP
Stat gains: R2os = -0.015106, MFSEadj = 0.640065, MFSEpvalue = 0.522325
Inference: 无样本外预测能力
Econ Gains: Delta U = -0.000170, Upred = 0.003902, Umean = 0.004072
Inference: 没有经济意义
---------------------------------------------------------------------

Out-of-sample tests for one factor model with OLS:
Predictor: VOL
Stat gains: R2os = 0.004113, MFSEadj = 2.097376, MFSEpvalue = 0.036293
Inference: 在5%的显著性水平下有样本外预测能力
Econ Gains: Delta U = -0.000069, Upred = 0.004003, Umean = 0.004072
Inference: 没有经济意义
---------------------------------------------------------------------

Out-of-sample tests for one factor model with O

In [13]:
# 样本外检验
# 多因子模型：OLS线性拟合
factor_out = 'DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6'
datafit = data.copy(deep=True)

n_in = np.sum(datafit['yyyymm'] <= 195612)
n_out = np.sum(datafit['yyyymm'] > 195612)
rout = np.zeros(n_out)
rmean = np.zeros(n_out)
rreal = np.zeros(n_out)
rfree = np.zeros(n_out)
volt2 = np.zeros(n_out)

for i in range(n_out):
    model = smf.ols('ExRet ~ DPL1 + EPL1 + VOLL1 + BILLL1 + BONDL1 + TERML1 + CREDITL1 + '
                    'PPIGL1 + IPGL1 + MA112L1 + MA312L1 + MOM6L1',
                    data=datafit[['ExRet', 'DPL1', 'EPL1', 'VOLL1', 'BILLL1', 'BONDL1', 'TERML1',
                                  'CREDITL1', 'PPIGL1', 'IPGL1', 'MA112L1', 'MA312L1', 'MOM6L1']].iloc[:(n_in+i), :])
    results = model.fit()
    k = results.params.values
    f = datafit[['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG',
                 'IPG', 'MA112', 'MA312', 'MOM6']].iloc[n_in+i-1, :].values
    f = np.concatenate((np.array([1]), f))
    rreal[i] = datafit['ExRet'].iloc[n_in+i]
    rfree[i] = datafit['Rfree'].iloc[n_in+i]
    rout[i] = np.sum(k*f)
    rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
    volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)

print()
print('Out-of-sample tests for multi-factor model with OLS:')
print('Predictor: {:s}'.format(factor_out))
R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
del datafit


Out-of-sample tests for multi-factor model with OLS:
Predictor: DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6
Stat gains: R2os = -0.032711, MFSEadj = 2.848436, MFSEpvalue = 0.004513
Inference: 无样本外预测能力
Econ Gains: Delta U = 0.000494, Upred = 0.004566, Umean = 0.004072
Inference: 有经济意义


In [14]:
# 样本外检验
# 多因子模型：LASSO回归, Ridge回归，ElasticNet回归

#Ridge回归
factor_out = 'DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6'
factor_list = np.array(['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG', 'IPG', 'MA112', 'MA312', 'MOM6'])

datafit = data.copy(deep=True)

n_in = np.sum(datafit['yyyymm'] <= 195612)
n_out = np.sum(datafit['yyyymm'] > 195612)
rout = np.zeros(n_out)
rmean = np.zeros(n_out)
rreal = np.zeros(n_out)
rfree = np.zeros(n_out)
volt2 = np.zeros(n_out)

# reg = sklm.LassoCV(random_state=0, cv=10, fit_intercept=True, normalize=True, precompute='auto', copy_X=True, n_jobs=-1, max_iter=10**9, tol=10-6)
# reg_lasso = linear_model.LassoLarsCV(cv=10, fit_intercept=True, normalize=True, precompute='auto', copy_X=True, n_jobs=-1, max_iter=10000000)
reg = sklm.RidgeCV(cv=10, fit_intercept=True,normalize=True)
# reg = sklm.ElasticNetCV(random_state=0, cv=10, fit_intercept=True, normalize=True, precompute='auto', copy_X=True, n_jobs=-1, max_iter=10**9, tol=10-6)
for i in range(n_out):
    X = datafit[['DPL1', 'EPL1', 'VOLL1', 'BILLL1', 'BONDL1', 'TERML1',
                 'CREDITL1', 'PPIGL1', 'IPGL1', 'MA112L1', 'MA312L1', 'MOM6L1']].iloc[:(n_in+i), :].values
    y = datafit['ExRet'].iloc[:(n_in+i)].values
    reg.fit(X, y)
    # print(factor_list[np.abs(reg.coef_) != 0])
    k = np.concatenate((np.array([reg.intercept_]), reg.coef_))
    f = datafit[['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG',
                 'IPG', 'MA112', 'MA312', 'MOM6']].iloc[n_in+i-1, :].values
    f = np.concatenate((np.array([1]), f))
    rreal[i] = datafit['ExRet'].iloc[n_in+i]
    rfree[i] = datafit['Rfree'].iloc[n_in+i]
    rout[i] = np.sum(k*f)
    rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
    volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)

print()
print('Out-of-sample tests for multi-factor model with ML method:')
print('Predictor: {:s}'.format(factor_out))
R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
del datafit


Out-of-sample tests for multi-factor model with ML method:
Predictor: DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6
Stat gains: R2os = 0.014139, MFSEadj = 2.522708, MFSEpvalue = 0.011850
Inference: 在5%的显著性水平下有样本外预测能力
Econ Gains: Delta U = 0.000159, Upred = 0.004231, Umean = 0.004072
Inference: 有经济意义


In [16]:
#LASSO回归

factor_out = 'DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6'
factor_list = np.array(['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG', 'IPG', 'MA112', 'MA312', 'MOM6'])

datafit = data.copy(deep=True)

n_in = np.sum(datafit['yyyymm'] <= 195612)
n_out = np.sum(datafit['yyyymm'] > 195612)
rout = np.zeros(n_out)
rmean = np.zeros(n_out)
rreal = np.zeros(n_out)
rfree = np.zeros(n_out)
volt2 = np.zeros(n_out)

reg = sklm.LassoCV(random_state=0, cv=10, fit_intercept=True,normalize=True,precompute='auto', copy_X=True, n_jobs=-1, max_iter=10**9, tol=10-6)
# reg_lasso = linear_model.LassoLarsCV(cv=10, fit_intercept=True, normalize=True, precompute='auto', copy_X=True, n_jobs=-1, max_iter=10000000)
for i in range(n_out):
    X = datafit[['DPL1', 'EPL1', 'VOLL1', 'BILLL1', 'BONDL1', 'TERML1',
                 'CREDITL1', 'PPIGL1', 'IPGL1', 'MA112L1', 'MA312L1', 'MOM6L1']].iloc[:(n_in+i), :].values
    y = datafit['ExRet'].iloc[:(n_in+i)].values
    reg.fit(X, y)
    # print(factor_list[np.abs(reg.coef_) != 0])
    k = np.concatenate((np.array([reg.intercept_]), reg.coef_))
    f = datafit[['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG',
                 'IPG', 'MA112', 'MA312', 'MOM6']].iloc[n_in+i-1, :].values
    f = np.concatenate((np.array([1]), f))
    rreal[i] = datafit['ExRet'].iloc[n_in+i]
    rfree[i] = datafit['Rfree'].iloc[n_in+i]
    rout[i] = np.sum(k*f)
    rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
    volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)

print()
print('Out-of-sample tests for multi-factor model with ML method:')
print('Predictor: {:s}'.format(factor_out))
R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
del datafit


Out-of-sample tests for multi-factor model with ML method:
Predictor: DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6
Stat gains: R2os = 0.009689, MFSEadj = 2.550774, MFSEpvalue = 0.010945
Inference: 在5%的显著性水平下有样本外预测能力
Econ Gains: Delta U = 0.000095, Upred = 0.004167, Umean = 0.004072
Inference: 有经济意义


In [17]:
# ElasticNet回归

factor_out = 'DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6'
factor_list = np.array(['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG', 'IPG', 'MA112', 'MA312', 'MOM6'])

datafit = data.copy(deep=True)

n_in = np.sum(datafit['yyyymm'] <= 195612)
n_out = np.sum(datafit['yyyymm'] > 195612)
rout = np.zeros(n_out)
rmean = np.zeros(n_out)
rreal = np.zeros(n_out)
rfree = np.zeros(n_out)
volt2 = np.zeros(n_out)

reg = sklm.ElasticNetCV(random_state=0, cv=10, fit_intercept=True,normalize=True,precompute='auto', copy_X=True, n_jobs=-1, max_iter=10**9, tol=10-6)
for i in range(n_out):
    X = datafit[['DPL1', 'EPL1', 'VOLL1', 'BILLL1', 'BONDL1', 'TERML1',
                 'CREDITL1', 'PPIGL1', 'IPGL1', 'MA112L1', 'MA312L1', 'MOM6L1']].iloc[:(n_in+i), :].values
    y = datafit['ExRet'].iloc[:(n_in+i)].values
    reg.fit(X, y)
    # print(factor_list[np.abs(reg.coef_) != 0])
    k = np.concatenate((np.array([reg.intercept_]), reg.coef_))
    f = datafit[['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG',
                 'IPG', 'MA112', 'MA312', 'MOM6']].iloc[n_in+i-1, :].values
    f = np.concatenate((np.array([1]), f))
    rreal[i] = datafit['ExRet'].iloc[n_in+i]
    rfree[i] = datafit['Rfree'].iloc[n_in+i]
    rout[i] = np.sum(k*f)
    rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
    volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)

print()
print('Out-of-sample tests for multi-factor model with ML method:')
print('Predictor: {:s}'.format(factor_out))
R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
del datafit


Out-of-sample tests for multi-factor model with ML method:
Predictor: DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6
Stat gains: R2os = 0.010282, MFSEadj = 2.563646, MFSEpvalue = 0.010551
Inference: 在5%的显著性水平下有样本外预测能力
Econ Gains: Delta U = 0.000095, Upred = 0.004167, Umean = 0.004072
Inference: 有经济意义


In [23]:
# 线性模型中的normalize参数已经被移除了，需要标准化化数据需要手动进行
# ElasticNet回归

from sklearn.preprocessing import StandardScaler
factor_out = 'DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6'
factor_list = np.array(['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG', 'IPG', 'MA112', 'MA312', 'MOM6'])

datafit = data.copy(deep=True)

n_in = np.sum(datafit['yyyymm'] <= 195612)
n_out = np.sum(datafit['yyyymm'] > 195612)
rout = np.zeros(n_out)
rmean = np.zeros(n_out)
rreal = np.zeros(n_out)
rfree = np.zeros(n_out)
volt2 = np.zeros(n_out)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(datafit[['DPL1', 'EPL1', 'VOLL1', 'BILLL1', 'BONDL1', 'TERML1','CREDITL1', 'PPIGL1', 'IPGL1', 'MA112L1', 'MA312L1', 'MOM6L1']])

reg = sklm.ElasticNetCV(random_state=0, cv=10, fit_intercept=True, precompute='auto', copy_X=True, n_jobs=-1, max_iter=10**9, tol=10-6)
for i in range(n_out):
    X = X_scaled[:(n_in+i),:]
    y = datafit['ExRet'].iloc[:(n_in+i)].values
    reg.fit(X, y)
    # print(factor_list[np.abs(reg.coef_) != 0])
    k = np.concatenate((np.array([reg.intercept_]), reg.coef_))
    f = datafit[['DP', 'EP', 'VOL', 'BILL', 'BOND', 'TERM', 'CREDIT', 'PPIG',
                 'IPG', 'MA112', 'MA312', 'MOM6']].iloc[n_in+i-1, :].values
    f = np.concatenate((np.array([1]), f))
    rreal[i] = datafit['ExRet'].iloc[n_in+i]
    rfree[i] = datafit['Rfree'].iloc[n_in+i]
    rout[i] = np.sum(k*f)
    rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
    volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)

print()
print('Out-of-sample tests for multi-factor model with ML method:')
print('Predictor: {:s}'.format(factor_out))
R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
del datafit


Out-of-sample tests for multi-factor model with ML method:
Predictor: DP, EP, VOL, BILL, BOND, TERM, CREDIT, PPIG, IPG, MA112, MA312, MOM6
Stat gains: R2os = -0.090962, MFSEadj = 0.973017, MFSEpvalue = 0.330857
Inference: 无样本外预测能力
Econ Gains: Delta U = -0.000683, Upred = 0.003390, Umean = 0.004072
Inference: 没有经济意义
